In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#Load dataset
df = pd.read_csv('/content/song_recomendation_B.csv')

#Head of dataset
df.head()

,Track URI,Track Name,Artist Name(s),Album Name,Album Release Date,Popularity,Artist Genres,Danceability,Energy,Key,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature
0,spotify:track:6a8GbQIlV8HBUW3c6Uk9PH,I Know You Want Me (Calle Ocho),Pitbull,Pitbull Starring In Rebelution,2009-10-23,64,"dance pop,miami hip hop,pop",0.825,0.743,2.0,-5.995,1.0,0.1490,0.0142,0.000021,0.2370,0.800,127.045,4.0
1,spotify:track:70XtWbcVZcpaOddJftMcVi,From the Bottom of My Broken Heart,Britney Spears,...Baby One More Time (Digital Deluxe Version),1999-01-12,56,"dance pop,pop",0.677,0.665,7.0,-5.171,1.0,0.0305,0.5600,NaN,0.3380,0.706,74.981,4.0
2,spotify:track:72WZtWs6V7uu3aMgMmEkYe,You Can't Always Get What You Want,The Rolling Stones,Let It Bleed,1969-12-05,0,"album rock,british invasion,classic rock,rock",0.319,0.627,0.0,-9.611,1.0,0.0687,0.6750,0.000073,0.2890,0.497,85.818,4.0
3,spotify:track:4bEb3KE4mSKlTFjtWJQBqO,Don't Stop - 2004 Remaster,Fleetwood Mac,Rumours,1977-02-04,79,"album rock,classic rock,rock,soft rock,yacht rock",0.671,0.710,9.0,-7.724,1.0,0.0356,0.0393,0.000011,0.0387,0.834,118.745,4.0
4,spotify:track:0d2iYfpKoM0QCKvcLCkBao,Eastside (with Halsey & Khalid),"benny blanco, Halsey, Khalid",Eastside (with Halsey & Khalid),2018-07-12,78,"pop,electropop,etherpop,indie poptimism,pop,po...",0.560,0.680,6.0,-7.648,0.0,0.3210,0.5550,0.000000,0.1160,0.319,89.391,4.0


In [ ]:
df.shape

(4999, 19)

Data terdiri dari 4999 baris dan 19 kolom.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Track URI           4999 non-null   object 
 1   Track Name          4999 non-null   object 
 2   Artist Name(s)      4999 non-null   object 
 3   Album Name          4999 non-null   object 
 4   Album Release Date  4997 non-null   object 
 5   Popularity          4999 non-null   int64  
 6   Artist Genres       4744 non-null   object 
 7   Danceability        4747 non-null   float64
 8   Energy              4997 non-null   float64
 9   Key                 4997 non-null   float64
 10  Loudness            4997 non-null   float64
 11  Mode                4997 non-null   float64
 12  Speechiness         4997 non-null   float64
 13  Acousticness        4997 non-null   float64
 14  Instrumentalness    4897 non-null   float64
 15  Liveness            4997 non-null   float64
 16  Valenc

# a. Conduct an Exploratory Data Analysis (EDA) on the dataset. Highlight the main insights and any irregularities you find, and address those anomalies appropriately!

## Check Data Problems

In [ ]:
#Check duplicate
df.duplicated().sum()

np.int64(13)

Terdapat 13 baris duplikat.

In [ ]:
#Handle duplicates
df = df.drop_duplicates()

Semua baris yang duplikat di drop agar analisis tidak bias dan tidak memengaruhi clustering.

In [ ]:
#Check Inconsistent
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].dropna().unique())


Column: Track URI
['spotify:track:6a8GbQIlV8HBUW3c6Uk9PH'
 'spotify:track:70XtWbcVZcpaOddJftMcVi'
 'spotify:track:72WZtWs6V7uu3aMgMmEkYe' ...
 'spotify:track:6PUzxtIHkv346yP89NzP9X'
 'spotify:track:3kcKlOkQQEPVwxwljbGJ5p'
 'spotify:track:5ydeCNaWDmFbu4zl0roPAH']

Column: Track Name
['I Know You Want Me (Calle Ocho)' 'From the Bottom of My Broken Heart'
 "You Can't Always Get What You Want" ... 'Kernkraft 400'
 'Kernkraft 400 (A Better Day)'
 "Groovejet (If This Ain't Love) [feat. Sophie Ellis-Bextor]"]

Column: Artist Name(s)
['Pitbull' 'Britney Spears' 'The Rolling Stones' ... 'Creeds'
 'Zombie Nation' 'Spiller, Sophie Ellis-Bextor']

Column: Album Name
['Pitbull Starring In Rebelution'
 '...Baby One More Time (Digital Deluxe Version)' 'Let It Bleed' ...
 'Kernkraft 400 Single Mixes' 'Kernkraft 400 (A Better Day)'
 "Groovejet (If This Ain't Love) [feat. Sophie Ellis-Bextor]"]

Column: Album Release Date
['2009-10-23' '1999-01-12' '1969-12-05' ... '2023-03-31' '2006-03-07'
 '2022-06-1

1. Key (kunci musik) sebenarnya adalah kategori, bukan angka, sehingga tidak cocok kalau diperlakukan seperti nilai numerik. Agar tidak menimbulkan perhitungan jarak yang salah, Key diubah menjadi format one-hot.
2. Mode menunjukkan apakah lagu bersifat mayor atau minor, sehingga sudah berupa data biner. Fitur ini tetap dipakai, tetapi distandarkan agar kontribusinya seimbang dengan fitur lainnya saat membandingkan kemiripan lagu.
3. Time Signature (jumlah ketukan per bar) ditemukan memiliki nilai nol yang tidak bermakna. Karena nilai nol tidak relevan secara musik dan fitur ini juga tidak terlalu berpengaruh pada kemiripan audio, maka variabel ini dihapus agar data lebih konsisten dan model lebih andal.

In [ ]:
#Check Missing Values
df.isna().sum()

,0
Track URI,0
Track Name,0
Artist Name(s),0
Album Name,0
Album Release Date,2
Popularity,0
Artist Genres,255
Danceability,252
Energy,2
Key,2


Kolom dengan Missing Value
1. Album Release Date -> 2 missing
2. Artist Genres -> 255 missing
3. Danceability -> 252 missing
4. Energy -> 2 missing
5. Key -> 2 missing
6. Loudness -> 2 missing
7. Mode -> 2 missing
8. Speechiness -> 2 missing
9. Acousticness -> 2 missing
10. Instrumentalness -> 102 missing
11. Liveness -> 2 missing
12. Valence -> 2 missing
13. Tempo -> 2 missing



In [ ]:
#Check Outliers
numeric_cols = [
    "Danceability", "Energy", "Loudness", "Speechiness",
    "Acousticness", "Instrumentalness", "Liveness",
    "Valence", "Tempo"
]

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]

    outlier_summary.append({
        'column': col,
        'outlier_count': outliers.shape[0],
        'outlier_min': outliers.min() if not outliers.empty else None,
        'outlier_max': outliers.max() if not outliers.empty else None
    })

outlier_df = pd.DataFrame(outlier_summary)

print(outlier_df[outlier_df['outlier_count'] > 0])

             column  outlier_count  outlier_min  outlier_max
0      Danceability             34      0.00000        0.228
1            Energy             25      0.00002        0.140
2          Loudness            103    -29.36800      -15.366
3       Speechiness            557      0.11800        0.711
4      Acousticness            231      0.78100        0.987
5  Instrumentalness            999      0.00134        0.973
6          Liveness            239      0.47700        0.982
8             Tempo            116      0.00000      213.654


1. Danceability (34 outlier, 0.000-0.228): Nilai sangat rendah menunjukkan lagu sangat tidak “danceable”. Bisa jadi memang karakter lagunya seperti itu atau data salah, jadi perlu dicek.
2. Energy (25 outlier, 0.00002-0.140): Energy yang sangat rendah biasanya berarti lagu sangat lembut atau minim dinamika. Bisa valid, tapi juga bisa error, jadi perlu dicek.
3. Loudness (103 outlier, -29.368-(-15.039)): Nilai loudness yang sangat rendah menunjukkan suara sangat pelan. Karena cukup banyak outlier, perlu dipertimbangkan untuk ditangani.
4. Speechiness (557 outlier, 0.118-0.711): Nilai tinggi berarti banyak suara bicara (misalnya podcast atau rap). Karena jumlah outlier sangat banyak, ini bisa memengaruhi hasil clustering.
5. Acousticness (231 outlier, 0.781-0.987): Menunjukkan lagu sangat akustik. Bisa valid, tapi karena banyak outlier, perlu dicek.
6. Instrumentalness (999 outlier, 0.00134-0.973): Outlier sangat banyak, menunjukkan banyak lagu instrumental atau data tidak konsisten. Perlu di-handle agar tidak memengaruhi model.
7. Liveness (239 outlier, 0.474-0.982): Nilai tinggi menunjukkan rekaman live (konser). Bisa valid, tapi tetap perlu dicek karena cukup banyak outlier.
8. Tempo (116 outlier, 0.000–213.654): Tempo 0 jelas tidak realistis, dan tempo sangat tinggi juga jarang. Jadi perlu ditangani.

## Handle Data Problems

In [ ]:
#Hanlde Irrelevant Value
df["Time Signature"] = df["Time Signature"].replace(0, np.nan)
df["Time Signature"].fillna(df["Time Signature"].median(), inplace=True)

Nilai 0 pada Time Signature tidak valid secara musik, jadi diganti menjadi missing agar tidak mempengaruhi analisis, lalu diisi dengan median untuk menjaga kelengkapan data.

In [ ]:
#Handle Missing Value
df = df.dropna()

Missng value didrop karena mengisi missing pada fitur audio penting bisa membuat data tidak realistis, sementara dataset masih cukup besar sehingga kehilangan beberapa baris tidak masalah.

In [ ]:
#Handle Outliers Use IQR Capping
outlier_cols = [
    "Danceability", "Energy", "Loudness", "Speechiness",
    "Acousticness", "Instrumentalness", "Liveness", "Tempo"
]

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # cap values
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])


Outlier pada fitur audio bisa valid karena beberapa lagu memang memiliki karakter ekstrem (misalnya instrumental atau live), namun nilai ekstrem ini dapat mengacaukan perhitungan kemiripan antar lagu dalam sistem rekomendasi berbasis konten karena distribusi fitur audio biasanya tidak normal, IQR capping lebih aman daripada z-score, dan metode ini menjaga data tetap utuh dengan hanya membatasi nilai ekstrem tanpa menghapus data.

In [ ]:
#One Hot Encoding
from sklearn.preprocessing import OneHotEncoder

df["Key"] = df["Key"].astype("Int64")
df["Time Signature"] = df["Time Signature"].astype("Int64")

ohe = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

encoded = ohe.fit_transform(df[["Key", "Time Signature"]])
encoded_cols = ohe.get_feature_names_out(["Key", "Time Signature"])
encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

df = pd.concat([df.drop(columns=["Key", "Time Signature"]), encoded_df], axis=1)

Variabel Key dan Time Signature pada dasarnya adalah kategori, bukan angka yang punya urutan atau jarak. Kalau dibiarkan sebagai angka, model bisa menganggap misalnya Key 1 lebih “dekat” dengan Key 2 dibanding Key 11, padahal secara musik tidak seperti itu. One Hot Encoding mengubah setiap kategori menjadi kolom terpisah sehingga model tidak menganggap ada urutan atau jarak antar kategori, sehingga perhitungan kemiripan atau model prediksi jadi lebih akurat.

In [ ]:
#Scaling
from sklearn.preprocessing import StandardScaler

num_cols = ["Danceability", "Energy", "Loudness", "Speechiness",
            "Acousticness", "Instrumentalness", "Liveness",
            "Valence", "Tempo"]

scaler = StandardScaler()
audio_scaled = scaler.fit_transform(df[num_cols])

Scaling dilakukan agar semua fitur audio memiliki skala yang sama, sehingga fitur dengan rentang nilai besar tidak mendominasi saat menghitung kemiripan atau saat dipakai dalam model. StandardScaler dipilih karena mengubah data menjadi distribusi dengan mean 0 dan standar deviasi 1, sehingga cocok untuk fitur yang memiliki skala dan distribusi berbeda seperti danceability, loudness, dan tempo.

# b. Create a content-based recommendation system, implemented as a function that takes a text input and returns five suggested items!

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#Text Feature
df["text_feature"] = df["Track Name"] + " " + df["Artist Name(s)"] + " " + df["Artist Genres"]

#TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["text_feature"])

In [ ]:
#Keyword Detection + threshold
def detect_keywords(text):
    text = text.lower()
    return {
        "fast": any(k in text for k in ["fast", "tempo", "upbeat", "quick"]),
        "slow": any(k in text for k in ["slow", "calm", "relax", "chill"]),
        "dance": any(k in text for k in ["dance", "edm", "club"]),
        "energetic": any(k in text for k in ["energetic", "powerful"]),
        "acoustic": "acoustic" in text,
        "instrumental": any(k in text for k in ["instrumental", "piano", "guitar"]),
        "rap": any(k in text for k in ["rap", "spoken", "hip hop"]),
        "happy": any(k in text for k in ["happy", "joy", "fun"]),
        "sad": any(k in text for k in ["sad", "melancholy", "emotional"]),
        "live": "live" in text
    }

def apply_threshold(df, keywords):
    mask = pd.Series(True, index=df.index)

    if keywords["fast"]:
        mask &= df["Tempo"] > 120
    if keywords["slow"]:
        mask &= df["Tempo"] < 90
    if keywords["dance"]:
        mask &= df["Energy"] > 0.6
    if keywords["energetic"]:
        mask &= df["Energy"] > 0.75
    if keywords["acoustic"]:
        mask &= df["Acousticness"] > 0.6
    if keywords["instrumental"]:
        mask &= df["Instrumentalness"] > 0.6
    if keywords["rap"]:
        mask &= df["Speechiness"] > 0.4
    if keywords["happy"]:
        mask &= df["Valence"] > 0.6
    if keywords["sad"]:
        mask &= df["Valence"] < 0.4
    if keywords["live"]:
        mask &= df["Liveness"] > 0.5

    return mask

In [ ]:
def recommend(text_input, top_n=5):
    keywords = detect_keywords(text_input)

    mask = apply_threshold(df, keywords)
    filtered_df = df[mask].copy()

    if len(filtered_df) < top_n:
        filtered_df = df.copy()

    tfidf_filtered = TfidfVectorizer(stop_words="english")
    tfidf_matrix_filtered = tfidf_filtered.fit_transform(filtered_df["text_feature"])

    audio_scaled_filtered = scaler.transform(filtered_df[num_cols])

    input_vec = tfidf_filtered.transform([text_input])
    text_sim = cosine_similarity(input_vec, tfidf_matrix_filtered).flatten()

    audio_sim = cosine_similarity(audio_scaled_filtered, audio_scaled_filtered)

    combined_scores = 0.5 * text_sim + 0.5 * np.mean(audio_sim, axis=1)

    top_indices = combined_scores.argsort()[-top_n:][::-1]

    return filtered_df.iloc[top_indices][["Track Name", "Artist Name(s)", "Artist Genres"]]

# c. Provide a minimum of three inputs to the function and examine if the recommendations correspond to each input!

In [ ]:
inputs = [
    "energetic dance pop",
    "calm acoustic guitar",
    "upbeat EDM track",
    "fast tempo dance track",
    "piano instrumental"
]

from IPython.display import display, Markdown

for i, inp in enumerate(inputs, 1):
    display(Markdown(f"### 🎧 Input {i}: *{inp}*"))

    rec = recommend(inp, top_n=5).reset_index(drop=True)
    rec.index += 1
    rec = rec.rename(columns={
        "Track Name": "Track",
        "Artist Name(s)": "Artist",
        "Artist Genres": "Genres"
    })

    display(rec)


### 🎧 Input 1: *energetic dance pop*

,Track,Artist,Genres
1,Lonely,Joel Corry,"dance pop,pop dance,uk dance"
2,Came Here for Love,"Sigala, Ella Eyre","dance pop,edm,pop dance,uk dance,uk pop,dance ..."
3,Never Gonna Not Dance Again,P!nk,"dance pop,pop"
4,I'm All Yours,"Jay Sean, Pitbull","dance pop,pop,pop rap,dance pop,miami hip hop,pop"
5,Love on Me,"Galantis, Hook N Sling","dance pop,edm,pop,pop dance,australian dance,p..."


### 🎧 Input 2: *calm acoustic guitar*

,Track,Artist,Genres
1,Calm Down,Killing Heidi,"australian alternative rock,australian rock"
2,Guitar Band,Stevie Wright,australian rock
3,Spanish Guitar,Toni Braxton,"contemporary r&b,dance pop,r&b,urban contemporary"
4,Collide - Acoustic Version,Howie Day,"acoustic pop,neo mellow,pop rock"
5,What Do You Mean? - Acoustic,Justin Bieber,"canadian pop,pop"


### 🎧 Input 3: *upbeat EDM track*

,Track,Artist,Genres
1,One Track Mind,Bobby Lewis,"doo-wop,rhythm and blues"
2,Light Years Away,"Tiësto, DBX","big room,brostep,dutch edm,edm,house,pop dance..."
3,The Nights,Avicii,"edm,pop,pop dance"
4,Come & Go (with Marshmello),"Juice WRLD, Marshmello","chicago rap,melodic rap,rap,brostep,edm,pop,pr..."
5,Red Lights,Tiësto,"big room,brostep,dutch edm,edm,house,pop dance..."


### 🎧 Input 4: *fast tempo dance track*

,Track,Artist,Genres
1,One Track Mind,Bobby Lewis,"doo-wop,rhythm and blues"
2,We Own It (Fast & Furious),"2 Chainz, Wiz Khalifa","atl hip hop,hip hop,pop rap,rap,southern hip h..."
3,With Ur Love (feat. Mike Posner),"Cher Lloyd, Mike Posner","dance pop,pop,post-teen pop,talent show,dance ..."
4,Magic Number,"The Potbelleez, B.o.B","aussietronica,australian dance,australian hous..."
5,If I Go,Ella Eyre,"dance pop,talent show,uk dance,uk pop"


### 🎧 Input 5: *piano instrumental*

,Track,Artist,Genres
1,Green Onions,Booker T. & the M.G.'s,"blues,classic soul,instrumental funk,instrumen..."
2,Soul Limbo,Booker T. & the M.G.'s,"blues,classic soul,instrumental funk,instrumen..."
3,Dance Wiv Me,Dizzee Rascal,"grime,instrumental grime"
4,Something I Need,OneRepublic,"piano rock,pop"
5,Love Runs Out,OneRepublic,"piano rock,pop"


## Kesimpulan

Sistem rekomendasi musik yang dibangun menggunakan pendekatan content-based recommendation dengan memanfaatkan kesamaan teks (text similarity) dan karakteristik audio dari setiap lagu. Deskripsi input pengguna diubah menjadi representasi vektor menggunakan TF-IDF, kemudian dihitung tingkat kemiripannya dengan data lagu menggunakan cosine similarity.

Untuk meningkatkan relevansi hasil rekomendasi, sistem juga menerapkan proses filtering berdasarkan kata kunci pada input pengguna, seperti dance, acoustic, EDM, fast tempo, dan instrumental. Kata kunci tersebut digunakan untuk menyaring lagu berdasarkan fitur audio (misalnya energy, tempo, acousticness, dan instrumentalness) serta genre, sehingga lagu yang tidak sesuai konteks dapat dieliminasi.

Hasil evaluasi menunjukkan bahwa setelah dilakukan perbaikan pada mekanisme filtering dan penghapusan fallback yang terlalu permisif, rekomendasi yang dihasilkan menjadi lebih konsisten dengan deskripsi input pengguna. Dengan demikian, sistem ini mampu memberikan rekomendasi musik yang lebih relevan dan mencerminkan preferensi pengguna secara lebih akurat.